# Kaggle Computer Vision PyTorch Template — 5 Hour Selection / No Internet

Template untuk image classification/regression.

Mendukung:
- CSV format: `train.csv` berisi filename/path + label
- Folder format: `train/class_name/image.jpg`
- PyTorch training loop
- default `USE_PRETRAINED=False` agar aman dari aturan external data
- valid split + best checkpoint + submission generator

Catatan umum:
- Template ini sengaja generic karena detail kompetisi belum diketahui.
- Auto-detect bisa salah. Bagian paling penting adalah cell `CONFIG`.
- Selalu cek `sample_submission.csv`, metric, dan format kolom sebelum final submit.
- Target pertama saat seleksi: buat `submission.csv` valid secepat mungkin.

In [ ]:
import os, re, gc, glob, math, time, random, warnings
from pathlib import Path
from pprint import pprint
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, mean_squared_error, mean_absolute_error, r2_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import transforms, models

RANDOM_STATE = 42

def seed_everything(seed=42):
    random.seed(seed); os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
seed_everything(RANDOM_STATE)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
print('Torch:', torch.__version__, 'Torchvision:', torchvision.__version__)

CONFIG = {
    'TRAIN_CSV': None,
    'TEST_CSV': None,
    'SAMPLE_SUBMISSION_FILE': None,
    'TRAIN_IMAGE_DIR': None,
    'TEST_IMAGE_DIR': None,
    'IMAGE_COL': None,
    'TARGET_COL': None,
    'ID_COL': None,
    # None, 'classification', 'regression'
    'TASK_TYPE': None,
    'IMAGE_SIZE': 224,
    'BATCH_SIZE': 32,
    'EPOCHS': 5,
    'LR': 1e-3,
    'WEIGHT_DECAY': 1e-4,
    'NUM_WORKERS': 2,
    'USE_PRETRAINED': False,
    'MODEL_NAME': 'resnet18',
    'VALID_SIZE': 0.2,
    'LIMIT_TRAIN_ROWS': None,
    'SUBMISSION_FILE': 'submission.csv',
    'BEST_MODEL_FILE': 'best_model.pth',
}

In [ ]:
INPUT_ROOT = Path('/kaggle/input')
if not INPUT_ROOT.exists(): INPUT_ROOT = Path('.')

csv_files = sorted(str(p) for p in INPUT_ROOT.glob('**/*.csv'))
image_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}
all_images = []
for ext in image_exts:
    all_images += [str(p) for p in INPUT_ROOT.glob(f'**/*{ext}')]
    all_images += [str(p) for p in INPUT_ROOT.glob(f'**/*{ext.upper()}')]
all_images = sorted(set(all_images))

print(f'Found {len(csv_files)} CSV files')
for i, f in enumerate(csv_files):
    try:
        prev = pd.read_csv(f, nrows=3)
        print(f'{i:02d}. {f} | columns={list(prev.columns)}')
    except Exception as e:
        print(f'{i:02d}. {f} | cannot preview: {e}')
print(f'Found {len(all_images)} images')
for f in all_images[:10]: print('-', f)

def pick_file(files, keywords):
    keywords = [k.lower() for k in keywords]
    for f in files:
        if any(k in Path(f).name.lower() for k in keywords): return f
    return None

CONFIG['TRAIN_CSV'] = CONFIG['TRAIN_CSV'] or pick_file(csv_files, ['train'])
CONFIG['TEST_CSV'] = CONFIG['TEST_CSV'] or pick_file(csv_files, ['test'])
CONFIG['SAMPLE_SUBMISSION_FILE'] = CONFIG['SAMPLE_SUBMISSION_FILE'] or pick_file(csv_files, ['sample', 'submission'])
pprint({k: CONFIG[k] for k in ['TRAIN_CSV', 'TEST_CSV', 'SAMPLE_SUBMISSION_FILE']})

In [ ]:
basename_to_path = {}
for p in all_images:
    basename_to_path.setdefault(Path(p).name, p)

def resolve_image_path(x, extra_roots=None):
    if pd.isna(x): return None
    x = str(x)
    p = Path(x)
    if p.exists(): return str(p)
    p2 = INPUT_ROOT / x
    if p2.exists(): return str(p2)
    if extra_roots:
        for root in extra_roots:
            if root is None: continue
            root = Path(root)
            for cand in [root / x, root / Path(x).name]:
                if cand.exists(): return str(cand)
    return basename_to_path.get(Path(x).name)

def detect_image_col(df):
    candidates = []
    for c in df.columns:
        if df[c].dtype == 'object':
            s = df[c].dropna().astype(str)
            if len(s) == 0: continue
            ext_ratio = s.str.lower().str.contains(r'\.(jpg|jpeg|png|bmp|webp|tif|tiff)$', regex=True).mean()
            name_score = int(any(k in c.lower() for k in ['image', 'img', 'file', 'path', 'filename']))
            if ext_ratio > 0.1 or name_score: candidates.append((c, name_score, ext_ratio))
    candidates = sorted(candidates, key=lambda x: (x[1], x[2]), reverse=True)
    return candidates[0][0] if candidates else None

def detect_target_col(train_df, test_df):
    diff = [c for c in train_df.columns if c not in test_df.columns]
    if len(diff) == 1: return diff[0]
    for c in ['target', 'label', 'class', 'category', 'species', 'y']:
        if c in train_df.columns and c not in test_df.columns: return c
    return train_df.columns[-1]

def detect_id_col(train_df, test_df, sample_df=None):
    common = [c for c in train_df.columns if c in test_df.columns]
    if sample_df is not None and sample_df.columns[0] in test_df.columns: return sample_df.columns[0]
    for c in ['id', 'ID', 'Id', 'image_id', 'filename', 'file_name']:
        if c in common: return c
    return None

In [ ]:
sample_submission = pd.read_csv(CONFIG['SAMPLE_SUBMISSION_FILE']) if CONFIG['SAMPLE_SUBMISSION_FILE'] else None

if CONFIG['TRAIN_CSV'] is not None and CONFIG['TEST_CSV'] is not None:
    print('Using CSV format')
    train_df = pd.read_csv(CONFIG['TRAIN_CSV'])
    test_df = pd.read_csv(CONFIG['TEST_CSV'])
    CONFIG['TARGET_COL'] = CONFIG['TARGET_COL'] or detect_target_col(train_df, test_df)
    CONFIG['ID_COL'] = CONFIG['ID_COL'] or detect_id_col(train_df, test_df, sample_submission)
    CONFIG['IMAGE_COL'] = CONFIG['IMAGE_COL'] or detect_image_col(train_df)
    target_col = CONFIG['TARGET_COL']; id_col = CONFIG['ID_COL']; image_col = CONFIG['IMAGE_COL']
    assert image_col is not None, "Set CONFIG['IMAGE_COL'] manual"
    roots = [CONFIG['TRAIN_IMAGE_DIR'], CONFIG['TEST_IMAGE_DIR']]
    train_df['image_path'] = train_df[image_col].map(lambda x: resolve_image_path(x, roots))
    test_df['image_path'] = test_df[image_col].map(lambda x: resolve_image_path(x, roots))
else:
    print('Trying folder classification format')
    parent_counts = {}
    for p in map(Path, all_images):
        parent_counts[p.parent.parent] = parent_counts.get(p.parent.parent, 0) + 1
    candidates = sorted(parent_counts.items(), key=lambda x: x[1], reverse=True)
    for c, n in candidates[:10]: print(c, n)
    assert candidates, 'No image folder found'
    train_root = Path(CONFIG['TRAIN_IMAGE_DIR']) if CONFIG['TRAIN_IMAGE_DIR'] else candidates[0][0]
    rows = []
    for class_dir in sorted([p for p in train_root.iterdir() if p.is_dir()]):
        for img in class_dir.iterdir():
            if img.suffix.lower() in image_exts:
                rows.append({'image_path': str(img), 'target': class_dir.name, 'id': img.name})
    train_df = pd.DataFrame(rows)
    target_col = 'target'; id_col = 'id'; image_col = 'image_path'
    if sample_submission is not None:
        test_df = sample_submission[[sample_submission.columns[0]]].copy()
        id_col = sample_submission.columns[0]
        test_df['image_path'] = test_df[id_col].map(lambda x: resolve_image_path(x))
    else:
        test_imgs = [p for p in all_images if not str(p).startswith(str(train_root))]
        test_df = pd.DataFrame({'image_path': test_imgs})
        test_df['id'] = test_df['image_path'].map(lambda x: Path(x).name)
        id_col = 'id'
    CONFIG['TARGET_COL'] = target_col; CONFIG['ID_COL'] = id_col; CONFIG['IMAGE_COL'] = image_col

print('CONFIG columns:', {k: CONFIG[k] for k in ['IMAGE_COL', 'TARGET_COL', 'ID_COL']})
print('train_df:', train_df.shape, 'test_df:', test_df.shape)
print('Missing image paths train/test:', train_df['image_path'].isna().sum(), test_df['image_path'].isna().sum())
display(train_df.head())
display(test_df.head())

train_df = train_df.dropna(subset=['image_path']).reset_index(drop=True)
test_df = test_df.dropna(subset=['image_path']).reset_index(drop=True)
if CONFIG['LIMIT_TRAIN_ROWS'] is not None:
    train_df = train_df.sample(min(CONFIG['LIMIT_TRAIN_ROWS'], len(train_df)), random_state=RANDOM_STATE).reset_index(drop=True)

In [ ]:
target_col = CONFIG['TARGET_COL']; id_col = CONFIG['ID_COL']
y_raw = train_df[target_col]
if CONFIG['TASK_TYPE'] is not None:
    task_type = CONFIG['TASK_TYPE']
elif y_raw.dtype == 'object' or str(y_raw.dtype).startswith('category') or y_raw.nunique() <= 30:
    task_type = 'classification'
else:
    task_type = 'regression'
print('TASK_TYPE:', task_type)

if task_type == 'classification':
    label_encoder = LabelEncoder()
    train_df['target_encoded'] = label_encoder.fit_transform(y_raw.astype(str))
    n_outputs = len(label_encoder.classes_)
    print('Classes:', list(label_encoder.classes_))
    display(train_df[target_col].value_counts().head(20))
else:
    label_encoder = None
    train_df['target_encoded'] = pd.to_numeric(y_raw, errors='coerce')
    n_outputs = 1
    display(train_df['target_encoded'].describe())

In [ ]:
if task_type == 'classification':
    counts = train_df['target_encoded'].value_counts()
    stratify = train_df['target_encoded'] if counts.min() >= 2 else None
else:
    stratify = None

tr_df, va_df = train_test_split(train_df, test_size=CONFIG['VALID_SIZE'], random_state=RANDOM_STATE, stratify=stratify)
tr_df = tr_df.reset_index(drop=True); va_df = va_df.reset_index(drop=True)
print('train split:', tr_df.shape, 'valid split:', va_df.shape)

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((CONFIG['IMAGE_SIZE'], CONFIG['IMAGE_SIZE'])),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
valid_transform = transforms.Compose([
    transforms.Resize((CONFIG['IMAGE_SIZE'], CONFIG['IMAGE_SIZE'])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class ImageDataset(Dataset):
    def __init__(self, df, transform=None, has_target=True):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.has_target = has_target
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            img = Image.open(row['image_path']).convert('RGB')
        except Exception:
            img = Image.new('RGB', (CONFIG['IMAGE_SIZE'], CONFIG['IMAGE_SIZE']))
        if self.transform: img = self.transform(img)
        if self.has_target:
            if task_type == 'classification':
                y = torch.tensor(row['target_encoded'], dtype=torch.long)
            else:
                y = torch.tensor(row['target_encoded'], dtype=torch.float32)
            return img, y
        return img

train_loader = DataLoader(ImageDataset(tr_df, train_transform, True), batch_size=CONFIG['BATCH_SIZE'], shuffle=True, num_workers=CONFIG['NUM_WORKERS'], pin_memory=True)
valid_loader = DataLoader(ImageDataset(va_df, valid_transform, True), batch_size=CONFIG['BATCH_SIZE']*2, shuffle=False, num_workers=CONFIG['NUM_WORKERS'], pin_memory=True)
test_loader = DataLoader(ImageDataset(test_df, valid_transform, False), batch_size=CONFIG['BATCH_SIZE']*2, shuffle=False, num_workers=CONFIG['NUM_WORKERS'], pin_memory=True)

images, targets = next(iter(train_loader))
print(images.shape, targets.shape, targets[:10])

In [ ]:
def build_model(model_name, n_outputs, pretrained=False):
    weights = None
    try:
        if model_name == 'resnet18':
            if pretrained:
                try: weights = models.ResNet18_Weights.DEFAULT
                except Exception: weights = None
            model = models.resnet18(weights=weights)
            model.fc = nn.Linear(model.fc.in_features, n_outputs)
        elif model_name == 'resnet34':
            if pretrained:
                try: weights = models.ResNet34_Weights.DEFAULT
                except Exception: weights = None
            model = models.resnet34(weights=weights)
            model.fc = nn.Linear(model.fc.in_features, n_outputs)
        elif model_name == 'efficientnet_b0':
            if pretrained:
                try: weights = models.EfficientNet_B0_Weights.DEFAULT
                except Exception: weights = None
            model = models.efficientnet_b0(weights=weights)
            model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, n_outputs)
        else:
            raise ValueError(model_name)
    except Exception as e:
        print('Model creation failed, fallback resnet18 random init:', repr(e))
        model = models.resnet18(weights=None)
        model.fc = nn.Linear(model.fc.in_features, n_outputs)
    return model

model = build_model(CONFIG['MODEL_NAME'], n_outputs, CONFIG['USE_PRETRAINED']).to(DEVICE)
criterion = nn.CrossEntropyLoss() if task_type == 'classification' else nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['LR'], weight_decay=CONFIG['WEIGHT_DECAY'])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, CONFIG['EPOCHS']))
scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == 'cuda'))
print(type(model).__name__)

In [ ]:
def train_one_epoch(model, loader):
    model.train(); total = 0; n = 0
    for images, targets in loader:
        images = images.to(DEVICE, non_blocking=True); targets = targets.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=(DEVICE == 'cuda')):
            out = model(images)
            loss = criterion(out.squeeze(1), targets) if task_type == 'regression' else criterion(out, targets)
        scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        bs = images.size(0); total += loss.item() * bs; n += bs
    return total / max(n, 1)

@torch.no_grad()
def validate(model, loader):
    model.eval(); total = 0; n = 0; ys = []; outs = []
    for images, targets in loader:
        images = images.to(DEVICE, non_blocking=True); targets = targets.to(DEVICE, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=(DEVICE == 'cuda')):
            out = model(images)
            loss = criterion(out.squeeze(1), targets) if task_type == 'regression' else criterion(out, targets)
        bs = images.size(0); total += loss.item() * bs; n += bs
        ys.append(targets.detach().cpu()); outs.append(out.detach().cpu())
    y_true = torch.cat(ys).numpy(); out = torch.cat(outs).numpy()
    metrics = {'loss': total / max(n, 1)}
    if task_type == 'classification':
        pred = out.argmax(axis=1)
        metrics['accuracy'] = accuracy_score(y_true, pred)
        metrics['f1_macro'] = f1_score(y_true, pred, average='macro')
    else:
        pred = out.squeeze()
        metrics['rmse'] = mean_squared_error(y_true, pred, squared=False)
        metrics['mae'] = mean_absolute_error(y_true, pred)
        metrics['r2'] = r2_score(y_true, pred)
    return metrics

def score_for_best(metrics):
    return metrics.get('f1_macro', metrics.get('accuracy', -metrics.get('rmse', metrics['loss']))) if task_type == 'classification' else -metrics.get('rmse', metrics['loss'])

In [ ]:
best_score = -1e18; best_epoch = -1
for epoch in range(1, CONFIG['EPOCHS'] + 1):
    start = time.time()
    tr_loss = train_one_epoch(model, train_loader)
    metrics = validate(model, valid_loader)
    scheduler.step()
    score = score_for_best(metrics)
    msg = f"Epoch {epoch}/{CONFIG['EPOCHS']} train_loss={tr_loss:.5f}"
    for k, v in metrics.items(): msg += f' val_{k}={v:.5f}'
    msg += f' time={time.time()-start:.1f}s'
    print(msg)
    if score > best_score:
        best_score = score; best_epoch = epoch
        torch.save(model.state_dict(), CONFIG['BEST_MODEL_FILE'])
        print('  saved best model')
print('Best epoch:', best_epoch, 'best_score:', best_score)
model.load_state_dict(torch.load(CONFIG['BEST_MODEL_FILE'], map_location=DEVICE))
model.to(DEVICE)

In [ ]:
@torch.no_grad()
def predict(model, loader):
    model.eval(); preds = []
    for images in loader:
        images = images.to(DEVICE, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=(DEVICE == 'cuda')):
            out = model(images)
        if task_type == 'classification':
            preds.append(torch.softmax(out, dim=1).detach().cpu().numpy())
        else:
            preds.append(out.squeeze(1).detach().cpu().numpy())
    return np.concatenate(preds, axis=0)

test_pred = predict(model, test_loader)
print('test_pred:', test_pred.shape)

In [ ]:
def make_submission():
    if sample_submission is not None:
        sub = sample_submission.copy()
        pred_cols = [c for c in sub.columns if c != sub.columns[0]]
    else:
        sub = pd.DataFrame()
        if id_col is not None and id_col in test_df.columns:
            sub[id_col] = test_df[id_col].values
        else:
            sub['id'] = test_df['image_path'].map(lambda x: Path(x).name)
        pred_cols = ['target']; sub['target'] = 0
    if task_type == 'regression':
        sub[pred_cols[0]] = test_pred
    else:
        if len(pred_cols) == test_pred.shape[1]:
            for i, c in enumerate(pred_cols): sub[c] = test_pred[:, i]
        else:
            lbl = np.argmax(test_pred, axis=1)
            sub[pred_cols[0]] = label_encoder.inverse_transform(lbl)
            # For binary probability submission:
            # if test_pred.shape[1] == 2: sub[pred_cols[0]] = test_pred[:, 1]
    return sub

submission = make_submission()
display(submission.head())
print('shape:', submission.shape, 'NaN:', submission.isna().sum().sum())
submission.to_csv(CONFIG['SUBMISSION_FILE'], index=False)
print('Saved:', CONFIG['SUBMISSION_FILE'])

## CV quick improvement list

- Jika GPU ada dan dataset kecil, naikkan `EPOCHS` ke 10–15.
- Jika allowed, coba `USE_PRETRAINED=True`; kalau gagal download/cache, balik ke `False`.
- Kalau class imbalance, buat weighted `CrossEntropyLoss`.
- Kalau binary metric AUC/logloss, submit probability positif, bukan label.
- Kalau gambar detail, coba `IMAGE_SIZE=320`, tapi cek runtime.